# The Baseline RAG Pipeline

---

This notebook is your **starting point**. The pipeline is already built and working: run it, explore its outputs, question it, and find its limits.

**How to use this notebook**

| | |
|---|---|
| 📖 **Read** | The explanations use plain language, no coding background needed |
| ▶️ **Run** | Execute cells top to bottom with Shift+Enter to see the pipeline in action |
| 💬 **Discuss** | Talk about the outputs with your peers, do they make sense? Would you trust them? |
| 🔧 **Experiment** | Modify queries, tweak parameters, break things on purpose |

> You don't need to understand every line of code. Focus on what the system gets right and wrong and on thinking about how this would apply in your own context.

---

## What This Notebook Covers

**Retrieval-Augmented Generation (RAG)** combines an AI assistant with a search capability across a corpus of documents. Instead of the AI making things up from memory, it first searches your documents and then answers based on what it finds. The answer can always be traced back to a source.

```
Your question  ->  Search your documents  ->  AI answers using only those documents
```

| Notebook | Focus |
|---|---|
| **Baseline (this notebook)** | Working baseline prototype |
| Feature Track 1 | How to measure answer quality |
| Feature Track 2 | Reliable, structured outputs |
| Feature Track 3 | Better retrieval strategies |
| Feature Track 4 | Multi-step agent workflows |

---

## Why RAG? The Problem with a Standalone LLM

### The Scenario
**PrimePack AG** buys packaging materials (pallets, cardboard boxes, tape) from multiple suppliers. Sustainability claims are increasingly scrutinised by customers and regulators. Employees need to answer questions like:
> *"What is the GWP of the Logypal 1 pallet, and is the figure verified?"*  
> *"Can we tell a customer that the tesa tape is PFAS-free?"*  
> *"Which of our suppliers have a certified EPD?"*

### Why Not Just Ask ChatGPT?
A general-purpose LLM has three fundamental problems for this task:

| Problem | Why It Matters |
|---|---|
| **Internal document** | LLMs don't know about internal company documents. |
| **Hallucination** | When asked about unknown products the LLM invents plausible-sounding but false figures. |
| **No evidence trail** | Even when correct, a raw LLM answer cannot be traced back to a source document. |

### The RAG Solution
RAG adds a **retrieval step** between the user's question and the LLM:

```
 Documents ──► Chunker ──► Embedder ──► Vector DB
                                              │
 User query ─────────────────► Embedder ─────►  Retriever ──► Top-k Chunks
                                                                      │
                                                               LLM + Prompt
                                                                      │
                                                               Answer + Sources
```

The LLM only sees documents that are **actually in the corpus**. The answer can be traced to specific source chunks. If the corpus does not contain the answer, the LLM is instructed to say so.

### What RAG Does *Not* Fix
RAG shifts the problem from hallucination to **retrieval quality**. If the right chunk is not retrieved, the answer will still be wrong (or absent). The later feature tracks address exactly this: better chunking, better retrieval, and better output structure.

---

## Core Concepts

### Chunks

A **chunk** is a short excerpt from a source document, a section of a PDF, one sheet of a spreadsheet, or one heading-delimited paragraph of a Markdown file. Chunks are the unit of indexing and retrieval.

```python
@dataclass
class Chunk:
    id: str           # unique identifier
    title: str        # e.g. section heading
    content: str      # the text that gets embedded
    metadata: dict    # source_file, page, ...
```

### Embeddings
An **embedding** converts text to a dense numeric vector (e.g. 384 dimensions). Semantically similar texts produce similar vectors. Here we use `all-MiniLM-L6-v2`, a compact local model that runs without an API key.

### Vector Store (ChromaDB)
A **vector store** persists chunk embeddings on disk and supports approximate nearest-neighbour search. Given a query embedding, it returns the `top_k` most similar chunks in milliseconds.

### Retriever
A **retriever** wraps a vector store and exposes a single `retrieve(query)` method. The baseline uses a `VectorStoreRetriever` with `top_k=5`.

### RAG Agent
The **RAG agent** combines a retriever and an LLM. Its `answer()` method:
1. Embeds the query
2. Retrieves the top-k chunks
3. Formats chunks as XML `<source>` tags in the prompt
4. Calls the LLM and returns the answer + cited sources

---

## Setup

**Prerequisites:** `conversational-toolkit` and `backend` must be installed in editable mode (`pip install -e conversational-toolkit/ && pip install -e backend/`). For the **Ollama** backend, start `ollama serve` and pull the model (`ollama pull mistral-nemo:12b`). For **OpenAI**, set `OPENAI_API_KEY` in your environment.

In [1]:
from pathlib import Path
import numpy as np
import os

from conversational_toolkit.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    inspect_chunks,
    build_vector_store,
    inspect_retrieval,
    build_agent,
    build_llm,
    ask,
    DATA_DIR,
    VS_PATH,
    RETRIEVER_TOP_K,
)

EMBEDDING_MODEL = (
    "text-embedding-3-small"  # (OpenAI), or "sentence-transformers/all-MiniLM-L6-v2"
)

# Choose your LLM backend: "ollama" (local, requires `ollama serve`) or "openai" (requires OPENAI_API_KEY)
BACKEND = "ollama"  # set this before running

if not BACKEND:
    raise ValueError(
        'BACKEND is not set. Edit the line above and set it to "ollama", or "openai".\n'
        "See Renku_README.md for setup instructions."
    )

ROOT = Path().resolve().parents[1]
print(f"Project root : {ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Vector store : {VS_PATH}")
print(f"LLM backend  : {BACKEND}")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Project root : /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag
Data dir     : /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data
Vector store : /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/backend/data_vs.db
LLM backend  : ollama


---

## Before the RAG Pipeline: The LLM on Its Own

A **large language model (LLM)** is a neural network trained on billions of words of text. It can summarise documents, answer questions, and generate structured output, but only from knowledge baked into its weights during training. It has no direct access to your internal documents.

Before building the RAG pipeline, let's interact with the LLM directly to understand what it can and cannot do on its own.

In [2]:
from conversational_toolkit.llms.base import LLMMessage, Roles

llm_standalone = build_llm(backend=BACKEND)

# A question the LLM can answer from general training data
general_question = "What does GWP stand for, and what unit is it typically measured in?"

response_general = await llm_standalone.generate(
    [LLMMessage(role=Roles.USER, content=general_question)]
)
print("---------------------------")
print(f"Q: {general_question}\n")
print("---------------------------")
print(f"A: {response_general.content}")

2026-03-02 08:40:15.330 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_llm:159 - LLM backend: Ollama (mistral-nemo:12b)
2026-03-02 08:40:15.353 | DEBUG    | conversational_toolkit.llms.ollama:__init__:60 - Ollama LLM loaded: mistral-nemo:12b; temperature: 0.3; seed: 42; tools: None; response_format: None
2026-03-02 08:40:43.567 | DEBUG    | conversational_toolkit.llms.ollama:generate:73 - Completion: model='mistral-nemo:12b' created_at='2026-03-02T07:40:43.542554Z' done=True done_reason='stop' total_duration=28167274917 load_duration=6464238292 prompt_eval_count=19 prompt_eval_duration=7118775958 eval_count=209 eval_duration=14491901842 message=Message(role='assistant', content='GWP stands for Global Warming Potential. It\'s a metric used to measure the impact of greenhouse gases on global warming over time compared to carbon dioxide (CO2). Here\'s how it works:\n\n1. **Definition**: GWP compares the warming effect of different greenhouse gases with CO2 serving as

---------------------------
Q: What does GWP stand for, and what unit is it typically measured in?

---------------------------
A: GWP stands for Global Warming Potential. It's a metric used to measure the impact of greenhouse gases on global warming over time compared to carbon dioxide (CO2). Here's how it works:

1. **Definition**: GWP compares the warming effect of different greenhouse gases with CO2 serving as the reference gas.

2. **Measurement Unit**: The unit for Global Warming Potential is "GWP units." It's dimensionless, meaning it doesn't have a specific unit like grams or kilograms. Instead, it's calculated based on the mass of the gas being measured and its warming effect over a certain time frame (usually 100 years).

Here are some examples of GWP values:

- CO2: 1 (by definition)
- Methane (CH4): ~25
- Nitrous Oxide (N2O): ~300
- Hydrofluorocarbons (HFCs) and Perfluorocarbbons (PFCs): vary widely but can be thousands to millions


The LLM answers that correctly, GWP is a well-known concept covered in its training data.

Now ask something specific to PrimePack AG's product portfolio:

In [3]:
# A product-specific question the LLM has never seen in training
primepack_question = "What is the Global Warming Potential (GWP) of the Logypal 1 pallet sold by PrimePack AG, and is the figure third-party verified? Provide the link to PrimePack AG's official website."

response_pp = await llm_standalone.generate(
    [LLMMessage(role=Roles.USER, content=primepack_question)]
)
print("---------------------------")
print(f"Q: {primepack_question}\n")
print("---------------------------")
print(f"A: {response_pp.content}")

2026-03-02 08:41:12.839 | DEBUG    | conversational_toolkit.llms.ollama:generate:73 - Completion: model='mistral-nemo:12b' created_at='2026-03-02T07:41:12.828286Z' done=True done_reason='stop' total_duration=23391354625 load_duration=87233708 prompt_eval_count=48 prompt_eval_duration=659473375 eval_count=330 eval_duration=22544690871 message=Message(role='assistant', content='I couldn\'t find a specific product called "Logypal 1" on PrimePack AG\'s official website (<https://www.primepack.ch/en/>), but I did find information about their products in general. Here\'s what I found regarding their Global Warming Potential (GWP) and third-party verification:\n\n1. **Global Warming Potential (GWP)**: PrimePack AG states that they use materials with low GWP values, such as cyclopentane or CO2, for their insulated panels. However, they don\'t provide specific GWP figures for each product on their website.\n\n   - Cyclopentane has a GWP of 300 compared to CO2 (1), which is significantly lower t

---------------------------
Q: What is the Global Warming Potential (GWP) of the Logypal 1 pallet sold by PrimePack AG, and is the figure third-party verified? Provide the link to PrimePack AG's official website.

---------------------------
A: I couldn't find a specific product called "Logypal 1" on PrimePack AG's official website (<https://www.primepack.ch/en/>), but I did find information about their products in general. Here's what I found regarding their Global Warming Potential (GWP) and third-party verification:

1. **Global Warming Potential (GWP)**: PrimePack AG states that they use materials with low GWP values, such as cyclopentane or CO2, for their insulated panels. However, they don't provide specific GWP figures for each product on their website.

   - Cyclopentane has a GWP of 300 compared to CO2 (1), which is significantly lower than other commonly used refrigerants like R410A (2088) or R507 (3985).
   - CO2, which they also use, has a GWP of 1.

2. **Third-party verifi

PrimePack AG is a fictional company, no training data exists for this product
- If the model gave a specific figure or website link: it is hallucinated.
- If the model said 'I don't know': that is honest, but still not useful.

Either way, the LLM cannot provide the actual figure with a verifiable source.

### Why do different models behave differently?

**OpenAI models** (GPT-4o, GPT-4o-mini) are extensively trained with human feedback (Reinforcement Learning from Human Feedback (RLHF)) to decline when they lack reliable information. For a fictional company like PrimePack AG, with no public web presence, the model has learned to say "I don't know" rather than confabulate a specific figure.

**Smaller local models** (Mistral, LLaMA 7 - 13 B) are typically less safety-fine-tuned. Without the reinforcement signal that penalises confident wrong answers, they are more likely to generate a plausible-sounding but fabricated answer.

**The problem in either case:** "I don't know" and a hallucinated answer are equally useless to an employee who needs to respond to a CSRD audit. The correct response, *"The verified GWP is X kg CO₂e according to the 2023 EPD (source: EPD_pallet_relicyc_logypal1.pdf)"*, requires access to the actual document.

---

### Choosing a Backend: OpenAI API vs Local Models

Two LLM backends are available in this workshop. Both expose the same interface, switching requires changing a single variable.

#### Comparison

| | **OpenAI API** (`gpt-4o-mini`, `gpt-4o`, …) | **Ollama -> local** (`mistral-nemo:12b`, `llama3.2`, …) |
|---|---|---|
| **Data security** | Queries and document chunks are sent to OpenAI's servers. You can request zero-data-retention. | Everything stays on-premise. Nothing leaves the machine. Suitable for confidential documents without any external data agreements. |
| **Model capability** | State-of-the-art. Follows complex instructions reliably, structures output well, handles edge cases. `gpt-4o-mini` is the default for this workshop, it is much cheaper than `gpt-4o` with most of the capability for RAG tasks. | Smaller models (7 - 13 B parameters) are weaker on complex reasoning and strict rule-following. For straightforward retrieval and summarisation tasks the quality gap narrows considerably. |
| **Cost** | Per-token billing. A typical RAG query costs a fraction of a cent. See the cost estimation section below. | No API cost: you pay for hardware (CPU/GPU) and electricity. |
| **Setup** | One API key, no local hardware required | `ollama serve` + model download |

#### Self-Hosting Larger Models

The quality gap between a 12 B local model and GPT-4o can be substantially closed at larger model sizes:

- **LLaMA 3.1 70 B, Mistral Large 2, Qwen 2.5 72 B**: run on GPUs. Quality can approach GPT-4o on structured tasks like RAG.
- **Quantised models (GGUF / GPTQ):** Reduce memory requirements with a modest quality trade-off, making larger models accessible on smaller hardware.

**For this workshop** `gpt-4o-mini` (OpenAI) and `mistral-nemo:12b` (Ollama) are both sufficient to demonstrate the full RAG pipeline.

---

### Cost Estimation

API costs scale with the number of tokens processed. **Input tokens** (your system prompt and retrieved document chunks) are cheaper than **output tokens** (the model's generated answer).

OpenAI API pricing for all models can be found [here](https://developers.openai.com/api/docs/pricing/). As an example, prices for `gpt-4o-mini` are:

| Token type | Price |
|---|---|
| Input | $0.15 / 1 M tokens |
| Output | $0.60 / 1 M tokens |

A rough rule of thumb: **1 token ≈ 4 characters** of English text.

In [4]:
from sme_kt_zh_collaboration_rag.feature0_ingestion import estimate_tokens

# Cost estimation for gpt-4o-mini
INPUT_PRICE_PER_TOKEN = 0.15 / 1_000_000  # USD
OUTPUT_PRICE_PER_TOKEN = 0.60 / 1_000_000  # USD


def estimate_cost(input_text: str, output_text: str) -> dict:
    input_tok = estimate_tokens(input_text)
    output_tok = estimate_tokens(output_text)
    cost = input_tok * INPUT_PRICE_PER_TOKEN + output_tok * OUTPUT_PRICE_PER_TOKEN
    return {"input_tokens": input_tok, "output_tokens": output_tok, "cost_usd": cost}


# Simulate a typical RAG query: system prompt + 5 retrieved chunks + user question --> short generated answer
example_input = (
    "You are a helpful AI assistant specialised in sustainability for PrimePack AG. "
    "Answer only using the provided document excerpts. Cite your sources.\n\n"
    "Source: EPD_pallet_relicyc_logypal1.pdf\n"
    "The Logypal 1 pallet has a declared GWP of 3.2 kg CO\u2082e per functional unit (A1\u2013A3), "
    "verified by an independent third-party auditor under ISO\u202014044.\n\n"
    "Source: ART_product_catalog.md\n"
    "The Logypal 1 (Product ID: 20-100) is a recycled-plastic pallet supplied by Relicyc. "
    "It is listed as the primary pallet for heavy-duty use in the PrimePack AG portfolio.\n\n"
    "[... 3 more retrieved chunks ...]\n\n"
    "Q: What is the GWP of the Logypal 1 pallet, and is it verified?"
)
example_output = (
    "The Logypal 1 pallet has a GWP of 3.2\u202fkg\u202fCO\u2082e per functional unit (A1\u2013A3), "
    "according to the third-party verified EPD (EPD_pallet_relicyc_logypal1.pdf). "
    "The figure has been independently audited under ISO\u202014044."
)

info = estimate_cost(example_input, example_output)
print(f"Input  : ~{info['input_tokens']:>5,} tokens (prompt + chunks + question)")
print(f"Output : ~{info['output_tokens']:>5,} tokens (generated answer)")
print(f"Cost   : ${info['cost_usd']:.6f} per query")
print()
print(f"At 1,000 queries / day     ->  ${info['cost_usd'] * 1_000:>8.4f} / day")
print(f"At 10,000 queries / day    ->  ${info['cost_usd'] * 10_000:>8.4f} / day")
print(f"At 1,000,000 queries / day ->  ${info['cost_usd'] * 1_000_000:>8.2f} / day")

Input  : ~  159 tokens (prompt + chunks + question)
Output : ~   52 tokens (generated answer)
Cost   : $0.000055 per query

At 1,000 queries / day     ->  $  0.0550 / day
At 10,000 queries / day    ->  $  0.5505 / day
At 1,000,000 queries / day ->  $   55.05 / day


The largest cost driver is context length. More retrieved chunks = more input tokens.
top_k=5 (~2,000 input tokens) is a reasonable starting point for RAG.

---
# RAG Pipeline

Now that we have seen what an LLM can and cannot do on its own, we are ready to build the retrieval layer that makes it genuinely useful. The following five steps walk through the full RAG pipeline end-to-end, from loading the documents all the way to a sourced answer.

```
 Documents ──► Chunker ──► Embedder ──► Vector DB
                                              │
 User query ─────────────────► Embedder ─────►  Retriever ──► Top-k Chunks
                                                                      │
                                                               LLM + Prompt
                                                                      │
                                                               Answer + Sources
```

---
## Step 1: Load and Chunk Documents

The `load_chunks()` function walks `data/`, converts each file to Markdown, and splits it into chunks using header-based splitting.

Files above 20 MB are flagged and skipped automatically.

> For a detailed look at parser options, chunking strategies, and chunk size analysis, see **feature0b_ingestion.ipynb**.


In [5]:
# Load documents from DATA_DIR and split them into chunks.
chunks = load_chunks(max_files=None)
# Print a statistical summary and sampled content for visual inspection.
inspect_chunks(chunks)

# Print size distribution
char_lengths = [len(c.content) for c in chunks]
over_limit = sum(1 for n in char_lengths if n > 1024)
over_limit_openai = sum(1 for n in char_lengths if n > 8192 * 4)
print(f"\nChunks total: {len(chunks)}")
print(f"Mean length (chars): {sum(char_lengths) // len(char_lengths)}")
print(f"Estimated to be over 256-token embedding limit: {over_limit} / {len(chunks)}")
print(
    f"Estimated to be over 8192-token embedding limit: {over_limit_openai} / {len(chunks)}"
)
print("\nSuccessfully loaded and chunked the documents!")

2026-03-02 08:41:22.305 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:199 - Skipping unsupported file type '': .DS_Store
2026-03-02 08:41:22.308 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:206 - Chunking 34 files from /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data
2026-03-02 08:41:26.565 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_internal_procurement_policy.pdf: 12 chunks
2026-03-02 08:41:28.807 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_logylight_incomplete_datasheet.pdf: 6 chunks
2026-03-02 08:41:30.158 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_product_catalog.pdf: 7 chunks
2026-03-02 08:41:30.169 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_product_overview.xlsx: 1 chunks
2026-03-02 08:41:31.766 | DEBUG    | sme_kt_zh_collaborat


Chunks total: 1663
Mean length (chars): 1414
Estimated to be over 256-token embedding limit: 611 / 1663
Estimated to be over 8192-token embedding limit: 3 / 1663

Successfully loaded and chunked the documents!


In [6]:
# Print 3 representative chunks
for c in chunks[:3]:
    print(f"--- [{c.metadata.get('source_file', '?')}] ---")
    print(f"Title  : {c.title!r}")
    print(f"Length : {len(c.content)} chars")
    print(f"Preview: {c.content[:200].strip()!r}")
    print()

--- [ART_internal_procurement_policy.pdf] ---
Title  : '## Supplier Sustainability Requirements'
Length : 202 chars
Preview: '## Supplier Sustainability Requirements\n\nVersion: 1.2 | Approved by CEO (Andrea Frei) | Effective: 1 January 2024 Classification: Internal use only, do not share externally without management approval'

--- [ART_internal_procurement_policy.pdf] ---
Title  : '## 1. Purpose and Scope'
Length : 475 chars
Preview: '## 1. Purpose and Scope\n\nThis document establishes the minimum sustainability requirements for all packaging product suppliers from whom PrimePack AG procures goods for resale or distribution. It appl'

--- [ART_internal_procurement_policy.pdf] ---
Title  : '## 2. Evidence Standards'
Length : 1331 chars
Preview: '## 2. Evidence Standards\n\nPrimePack AG classifies supplier sustainability claims into four evidence levels:\n\n| Level   | Label     | Definition'



---

## Step 2: Embed Chunks and Build the Vector Store

`SentenceTransformerEmbeddings` converts every chunk's `content` to a 384-dimensional vector using `all-MiniLM-L6-v2`. The resulting matrix (shape `[n_chunks, 384]`) is inserted into a persistent `ChromaDBVectorStore`.

**On subsequent runs**, leave `reset=False` (the default) to skip re-embedding, it takes time and the store on disk is already correct. Pass `reset=True` only when the corpus or chunking strategy changes.

---

### Embedding Models
Two model families are implemented in the toolkit. The choice affects retrieval quality, cost, context limit, and data security.

| | `all-MiniLM-L6-v2` (default) | Other SentenceTransformer models | `text-embedding-3-small` (OpenAI) |
|---|---|---|---|
| **Dimensions** | 384 | 384 - 1024 (model-dependent) | 1024 (toolkit setting) |
| **Context limit** | 256 tokens | 256 - 8 192 (model-dependent) | 8 191 tokens |
| **Cost** | Free, local | Free, local | ~$0.02 / 1 M tokens |
| **Data security** | Fully local | Fully local | Sent to OpenAI |
| **Quality** | Good for short, focused text | Varies; some match OpenAI | State-of-the-art |
| **Setup** | No API key | No API key | `OPENAI_API_KEY` required |

**SentenceTransformer is a free alternatives from HuggingFace**: Any model from HuggingFace that is compatible with the `sentence-transformers` library works with our `SentenceTransformerEmbeddings` by passing a different `model_name`. Browse quality rankings on the [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).

Hardware matters: larger models are slower on CPU. 

**OpenAI embeddings**
`OpenAIEmbeddings` from `conversational_toolkit.embeddings.openai` calls the OpenAI API. The toolkit requests 1024 dimensions using OpenAI's Matryoshka dimension reduction, a technique that allows truncating full embeddings to a smaller size with minimal quality loss. Requires `OPENAI_API_KEY`.

---

### Vector Store

A vector store persists chunk embeddings on disk and provides approximate nearest-neighbour search. Two implementations are in the toolkit.

**`ChromaDBVectorStore`** (used in this notebook)
- Embedded database: no separate server process, data stored as files on disk at `VS_PATH`.
- Survives session restarts, which is why `reset=False` is safe by default.
- Uses L2 distance for search. For unit-length vectors (which both embedding models produce) this gives the same ranking as cosine similarity -> different numbers, identical top-k order.
- Well-suited for corpora up to ~100 k chunks. Not designed for concurrent writes or multi-user access.

**`PGVectorStore`** (also in the toolkit)
- PostgreSQL with the `pgvector` extension. Requires a running Postgres instance.
- Uses cosine similarity natively (also supports other).
- Supports rich metadata filtering, concurrent reads and writes, and standard SQL queries alongside vector search.
- The right choice when you already have a Postgres infrastructure, need concurrent access, or want to combine vector search with relational data.

In [ ]:
if "sentence-transformers" in EMBEDDING_MODEL:
    embedding_model = SentenceTransformerEmbeddings(model_name=EMBEDDING_MODEL)
else:
    embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
print(f"Embedding model: {EMBEDDING_MODEL}")

# Set reset=True to rebuild the store from scratch
vector_store = await build_vector_store(
    chunks, embedding_model, db_path=VS_PATH, reset=True
)
print("Vector store ready.")

2026-03-02 08:48:52.202 | DEBUG    | conversational_toolkit.embeddings.openai:__init__:20 - OpenAI embeddings model loaded: text-embedding-3-small


Embedding model: text-embedding-3-small


2026-03-02 08:48:52.526 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:267 - Reset vector store collection at /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/backend/data_vs.db
2026-03-02 08:48:52.526 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:275 - Embedding 1663 chunks with 'text-embedding-3-small' ...
2026-03-02 08:49:09.739 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1663, 1024)
2026-03-02 08:49:09.741 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:279 - Embedding matrix: shape=(1663, 1024)  dtype=float64
2026-03-02 08:49:10.811 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_vector_store:282 - Done! Vector store written to /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/backend/data_vs.db


Vector store ready.


### Similarity in Embedding Space

Embeddings that are close in vector space share semantic meaning (for visualisation of embeddings in the vector space look at this [website](https://projector.tensorflow.org)). The cell below embeds several sentences and measures their cosine similarity: a value between -1 (opposite) and 1 (identical). You can change the sentences to see the impact on cosine similarity.

In [8]:
sentence1 = "carbon footprint of a pallet"
sentence2 = "GWP value for the Logypal 1"
sentence3 = "PFAS-free tape declaration"
sentence4 = "the annual report of a software firm"


async def cosine_similarity(a: str, b: str) -> float:
    vecs = await embedding_model.get_embeddings([a, b])
    return float(
        np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1]))
    )


pairs = [
    (sentence1, sentence2),
    (sentence1, sentence3),
    (sentence1, sentence4),
]

print("Cosine similarities:")
for a, b in pairs:
    sim = await cosine_similarity(a, b)
    print(f"{sim:.3f}  -->  {a!r}  vs  {b!r}")

Cosine similarities:


2026-03-02 08:49:31.035 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (2, 1024)
2026-03-02 08:49:31.189 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (2, 1024)


0.351  -->  'carbon footprint of a pallet'  vs  'GWP value for the Logypal 1'
0.297  -->  'carbon footprint of a pallet'  vs  'PFAS-free tape declaration'


2026-03-02 08:49:31.409 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (2, 1024)


0.198  -->  'carbon footprint of a pallet'  vs  'the annual report of a software firm'


### Comparing Embedding Models on our Documents

A quick way to compare models without running a full evaluation: pick a query, a relevant chunk, and an irrelevant chunk, then measure the **cosine similarity gap** -> how much more similar the relevant chunk is to the query than the irrelevant one. A larger gap means the model discriminates better between useful and noise results, which translates directly to higher retrieval precision.

The cell below runs this for three CPU-friendly models (and OpenAI if the key is set). The HuggingFace models are downloaded on first use (~400 MB each, takes a minute).

In [9]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Test query and two chunks from the corpus
query = "What is the carbon footprint of the Logypal 1 pallet?"

chunk_relevant = "Logypal 1  -  GWP: 3.2 kg CO2e per functional unit (A1-A3). Figure verified by independent third-party auditor under ISO 14044."
chunk_irrelevant = "PrimePack AG Supplier Code of Conduct. All suppliers must comply with applicable environmental regulations and report annually on progress."

# Models to compare (CPU-friendly by default)
models_to_compare: dict = {
    "all-MiniLM-L6-v2 (90 MB, 384-dim)": SentenceTransformerEmbeddings(
        "all-MiniLM-L6-v2"
    ),
    "all-mpnet-base-v2 (420 MB, 768-dim)": SentenceTransformerEmbeddings(
        "sentence-transformers/all-mpnet-base-v2"
    ),
    "multilingual-MiniLM-L12 (470 MB, 384-dim)": SentenceTransformerEmbeddings(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    ),
}
# Add OpenAI if key is available
if os.getenv("OPENAI_API_KEY"):
    models_to_compare["text-embedding-3-small  (API, 1024-dim)"] = OpenAIEmbeddings(
        "text-embedding-3-small"
    )

# Run comparison
print("---------------------------")
print(f"Query     : {query!r}")
print(f"Relevant  : {chunk_relevant[:70]!r}...")
print(f"Irrelevant: {chunk_irrelevant[:70]!r}...")
print()
print(f"{'Model':<44}  {'sim(relevant)':>13}  {'sim(irrelevant)':>15}  {'gap':>6}")
print("-" * 84)

for name, model in models_to_compare.items():
    vecs = await model.get_embeddings([query, chunk_relevant, chunk_irrelevant])
    sim_rel = cosine_sim(vecs[0], vecs[1])
    sim_irr = cosine_sim(vecs[0], vecs[2])
    print(f"{name:<44}  {sim_rel:>13.3f}  {sim_irr:>15.3f}  {sim_rel - sim_irr:>6.3f}")

print("\nGap = sim(relevant) - sim(irrelevant). Larger gap -> better discrimination.")

2026-03-02 08:49:37.032 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:__init__:54 - Sentence Transformer embeddings model loaded: all-MiniLM-L6-v2 with kwargs: {}
2026-03-02 08:49:39.804 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:__init__:54 - Sentence Transformer embeddings model loaded: sentence-transformers/all-mpnet-base-v2 with kwargs: {}
2026-03-02 08:49:42.974 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:__init__:54 - Sentence Transformer embeddings model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 with kwargs: {}
2026-03-02 08:49:42.993 | DEBUG    | conversational_toolkit.embeddings.openai:__init__:20 - OpenAI embeddings model loaded: text-embedding-3-small


---------------------------
Query     : 'What is the carbon footprint of the Logypal 1 pallet?'
Relevant  : 'Logypal 1  -  GWP: 3.2 kg CO2e per functional unit (A1-A3). Figure ver'...
Irrelevant: 'PrimePack AG Supplier Code of Conduct. All suppliers must comply with '...

Model                                         sim(relevant)  sim(irrelevant)     gap
------------------------------------------------------------------------------------


2026-03-02 08:49:43.410 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:65 - all-MiniLM-L6-v2 embeddings size: (3, 384)
2026-03-02 08:49:43.597 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:65 - sentence-transformers/all-mpnet-base-v2 embeddings size: (3, 768)


all-MiniLM-L6-v2 (90 MB, 384-dim)                     0.570            0.065   0.505
all-mpnet-base-v2 (420 MB, 768-dim)                   0.584            0.287   0.297


2026-03-02 08:49:43.637 | DEBUG    | conversational_toolkit.embeddings.sentence_transformer:get_embeddings:65 - sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 embeddings size: (3, 384)


multilingual-MiniLM-L12 (470 MB, 384-dim)             0.671            0.166   0.505


2026-03-02 08:49:44.009 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (3, 1024)


text-embedding-3-small  (API, 1024-dim)               0.606            0.295   0.311

Gap = sim(relevant) - sim(irrelevant). Larger gap -> better discrimination.


**Disclaimer**: This is a quick example, not a rigorous evaluation. A single (query, chunk) pair can be misleading -> one model may score higher here and worse on a different example. Reliable model selection requires testing across many diverse queries.

---

## Step 3: Inspect Retrieval (Before the LLM Sees Anything)

This is the **most important diagnostic step** in the whole pipeline:

> If the retrieved chunks are wrong, the final answer will be wrong regardless of how good the LLM is.

`inspect_retrieval()` runs the query through the embedding model, fetches the top-k most similar chunks from ChromaDB, and prints them with scores. Use this to verify that relevant documents are in the index, tune `top_k`, compare different query phrasings, and identify retrieval gaps before blaming the LLM.

The **similarity score** is the L2 distance, range [0,4], lower = more similar. L2 distance is used becuase it works for any vectors, normalised or not. Cosine similarity only makes sense for direction (magnitude doesn't matter), so it requires that vectors be unit-length to be meaningful. L2 makes no such assumption, making it the safer general default. ChromaDB defaults to L2 because it's simpler to compute and works even if vector magnitudes vary. When the embedding model produces equal-length vectors, we get cosine-equivalent ranking. The score numbers look different, but the top-5 results would be identical either way.

In [10]:
QUERY = "What materials is the Logypal 1 pallet made from?"

results = await inspect_retrieval(
    QUERY, vector_store, embedding_model, top_k=RETRIEVER_TOP_K
)

2026-03-02 08:50:08.849 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)
2026-03-02 08:50:08.895 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:inspect_retrieval:301 - Retrieval for query: 'What materials is the Logypal 1 pallet made from?'



Top-5 retrieved chunks (returned=5; showing a maximum of 1000 content characters):
  [1] score=0.4439  file='ART_relicyc_logypal1_datasheet_2021.pdf'  title='## Overview'
       "## Overview\n\nThe Logypal 1 is Relicyc's flagship pallet, manufactured from 100% post-consumer recycled plastic, primarily sourced from end-of-life agricultural packaging (silage film) and industrial packaging waste. It is designed as a direct drop-in replacement for standard EUR wood pallets (1200 × 800 mm)."
  [2] score=0.4725  file='EPD_pallet_relicyc_logypal1.pdf'  title='## Logypal 1 ®'
       '## Logypal 1 ®\n\nAll these pallets are produced with secondary raw materials (a mix of polypropylene and high density polyethylene).\n\nThese new plastic pallets are the real alternative to the ISPM-15 treated wooden pallet (HT standard phytosanitary treatment that certifies the suitability of the material to the international regulations drawn up by the IPPC), having a comparable cost, but without the bureaucra

### Retrieval for a Product Outside the Portfolio

The PrimePack AG product catalog defines the portfolio boundary. The **Lara Pallet** is not in the catalog, it does not exist. Watch which chunks are returned and what scores they have. A **higher** minimum score (large L2 distance) signals *weaker semantic match*.

In [11]:
QUERY_OOK = "What materials is the Lara pallet made from?"

results_ook = await inspect_retrieval(
    QUERY_OOK, vector_store, embedding_model, top_k=RETRIEVER_TOP_K
)

2026-03-02 08:50:12.947 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)
2026-03-02 08:50:12.952 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:inspect_retrieval:301 - Retrieval for query: 'What materials is the Lara pallet made from?'



Top-5 retrieved chunks (returned=5; showing a maximum of 1000 content characters):
  [1] score=0.8117  file='EPD_pallet_relicyc_logypal1.pdf'  title='## Logypal 1 ®'
       '## Logypal 1 ®\n\nAll these pallets are produced with secondary raw materials (a mix of polypropylene and high density polyethylene).\n\nThese new plastic pallets are the real alternative to the ISPM-15 treated wooden pallet (HT standard phytosanitary treatment that certifies the suitability of the material to the international regulations drawn up by the IPPC), having a comparable cost, but without the bureaucracy and mandatory certifications for purchase. These products are also light, resistant, washable and resistant to mold and humidity.\n\nThe main characteristics of the model of pallet under study are shown in the following table:\n\n| MODEL                                 | Logypal 1    | |---------------------------------------|--------------| | Dimensions [mm] (LxWxH)               | 1200x800x138 | | Wei

> **Observation:** The retriever always returns the *closest* chunks it can find, it has no concept of "no match". For an unknown product the L2 distances are **higher** (the closest chunks are still about other pallets), but the LLM receives those chunks anyway and may silently answer about the wrong product.

---

## Step 4: Build the RAG Agent

`build_agent()` assembles the three components:

```
VectorStoreRetriever
    └─ ChromaDBVectorStore (on disk, persists across runs)
    └─ Embeddings (SentenceTransformer or OpenAI)

RAG Agent
    ├─ LLM (Ollama / OpenAI)
    ├─ Retriever
    └─ System prompt
```

### The System Prompt

The system prompt is a key lever for controlling LLM behaviour. It is prepended to every conversation and defines the rules the model must follow:

```
You are a helpful AI assistant specialised in sustainability and product compliance
for PrimePack AG. 

You will receive document excerpts relevant to the user's question. Produce the best possible answer using only the information in those excerpts.
```

In [12]:
llm = build_llm(backend=BACKEND)

SYSTEM_PROMPT = (
    "You are a helpful AI assistant specialised in sustainability and product compliance for PrimePack AG.\n\n"
    "You will receive document excerpts relevant to the user's question. Produce the best possible answer using only the information in those excerpts."
)
agent = build_agent(
    vector_store=vector_store,
    embedding_model=embedding_model,
    llm=llm,
    top_k=RETRIEVER_TOP_K,
    system_prompt=SYSTEM_PROMPT,
    number_query_expansion=0,  # 0 = no expansion; see Feature Track 3 for more
)
print("RAG agent assembled.")

2026-03-02 08:50:17.376 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_llm:159 - LLM backend: Ollama (mistral-nemo:12b)
2026-03-02 08:50:17.401 | DEBUG    | conversational_toolkit.llms.ollama:__init__:60 - Ollama LLM loaded: mistral-nemo:12b; temperature: 0.3; seed: 42; tools: None; response_format: None
2026-03-02 08:50:17.401 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:build_agent:335 - RAG agent ready (top_k=5  query_expansion=0)


RAG agent assembled.


---

## Step 5: Ask a Question

`ask()` sends the query to the agent and returns the answer string. The internal flow is:

1. Embed the query
2. Retrieve top-k chunks
3. Build the prompt: `<system>` + `<sources>` XML block + user question
4. Generate the answer with the LLM
5. Return the answer and a list of cited source chunks

In [13]:
QUERY = "What materials is the Logypal 1 pallet made from?"

print("---------------------------")
print(f"Query: {QUERY!r}")
print("---------------------------")
answer = await ask(agent, QUERY)

2026-03-02 08:50:20.684 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:350 - Query: 'What materials is the Logypal 1 pallet made from?'
2026-03-02 08:50:20.851 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)


---------------------------
Query: 'What materials is the Logypal 1 pallet made from?'
---------------------------


2026-03-02 08:50:49.398 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:353 - Answer:


The Logypal 1 pallet is made from 100% post-consumer recycled plastic. The primary materials used are polypropylene (PP) and high density polyethylene (HDPE), which make up over 99% of the pallet's composition. These materials were primarily sourced from end-of-life agricultural packaging like silage film and industrial packaging waste.
Sources (5):
  'ART_relicyc_logypal1_datasheet_2021.pdf'  |  '## Overview'
  'EPD_pallet_relicyc_logypal1.pdf'  |  '## Logypal 1 ®'
  'SPEC_pallet_relicyc_logypal1.pdf'  |  '## Cod. LOGYPAL1'
  'EPD_pallet_relicyc_logypal1.pdf'  |  '## CONTENT DECLARATION'
  'ART_logylight_incomplete_datasheet.pdf'  |  '## Product Overview'


---

## Probing Failure Modes

The dataset was designed with three deliberate challenges. Run the queries below and observe the answers.

### a) Out-of-Portfolio Query

The **Lara Pallet** does not exist. A good RAG must say so instead of describing a different pallet.

In [14]:
QUERY = "What materials is the Lara pallet made from?"
print("---------------------------")
print(f"Query: {QUERY!r}")
print("---------------------------")
answer_ook = await ask(agent, QUERY)

2026-03-02 08:51:07.412 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:350 - Query: 'What materials is the Lara pallet made from?'


---------------------------
Query: 'What materials is the Lara pallet made from?'
---------------------------


2026-03-02 08:51:07.624 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)
2026-03-02 08:51:22.732 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:353 - Answer:


The Lara pallet, as mentioned in source "d8693700-109b-448b-a7e7-3d03856f9fea", is made from 100% post-consumer recycled plastic. The main material used is primarily sourced from end-of-life agricultural packaging and industrial packaging waste.
Sources (5):
  'EPD_pallet_relicyc_logypal1.pdf'  |  '## Logypal 1 ®'
  'ART_relicyc_logypal1_datasheet_2021.pdf'  |  '## Overview'
  'ART_supplier_brochure_CPR_wood_pallet.pdf'  |  '## Material Sourcing'
  'ART_supplier_brochure_CPR_wood_pallet.pdf'  |  '## Sustainable by Nature'
  'SPEC_pallet_CPR_recycled_plastic.pdf'  |  '## Features:'


### b) Missing Data (LogyLight Pallet)

The LogyLight datasheet marks all LCA fields as *"not yet available"*. The correct answer is that we don't have the data, not a fabricated figure or saying that there is no infromation on it.

In [15]:
QUERY = "What is the GWP of the LogyLight pallet?"
print("---------------------------")
print(f"Query: {QUERY!r}")
print("---------------------------")
answer_gap = await ask(agent, QUERY)

2026-03-02 08:51:38.845 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:350 - Query: 'What is the GWP of the LogyLight pallet?'


---------------------------
Query: 'What is the GWP of the LogyLight pallet?'
---------------------------


2026-03-02 08:51:39.124 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)
2026-03-02 08:52:07.988 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:353 - Answer:


Based on the provided sources, there is no direct GWP (Global Warming Potential) value given for the LogyLight pallet specifically in any of the excerpts. However, we can infer some information:

1. The Logypal 1, which is made from similar materials and has a similar design to the LogyLight, has a GWP of 4.1 kg CO₂e per pallet over its full lifecycle (50-trip expected lifetime) according to an internal LCA conducted by Relicyc in 2020.
2. The sources do not specify if there's any difference in environmental impact between Logypal 1 and LogyLight.

Given the similarity in materials and design, it's reasonable to assume that the GWP for the LogyLight would be similar to that of the Logypal 1, i.e., approximately **4.1 kg CO₂e per pallet** over its full lifecycle. However, please note that this is an inference based on similarity and not a direct value provided in the sources.

For precise data, it's recommended to consult Relicyc directly or await further documentation from them regardi

### c) Conflicting Evidence (Relicyc GWP Figures)

The 2021 Relicyc datasheet reports **4.1 kg CO₂e** per pallet. The 2023 EPD (third-party verified) reports a different, more recent figure. The RAG should flag the conflict and prefer the verified, more recent source.

In [16]:
QUERY = "What is the GWP of the Logypal 1 pallet, and how reliable is the figure?"
print("---------------------------")
print(f"Query: {QUERY!r}")
print("---------------------------")
answer_conflict = await ask(agent, QUERY)

2026-03-02 08:53:09.248 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:350 - Query: 'What is the GWP of the Logypal 1 pallet, and how reliable is the figure?'


---------------------------
Query: 'What is the GWP of the Logypal 1 pallet, and how reliable is the figure?'
---------------------------


2026-03-02 08:53:09.471 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)
2026-03-02 08:53:26.155 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:353 - Answer:


The GWP of the Logypal 1 pallet is 4.1 kg CO₂ eq/pallet, according to an internal lifecycle assessment conducted by Relicyc in 2020 using the SimaPro software and the ecoinvent 3.6 database. The figure has not been independently verified at the time of publication.
Sources (5):
  'EPD_pallet_relicyc_logypal1.pdf'  |  '## Logypal 1 ®'
  'ART_relicyc_logypal1_datasheet_2021.pdf'  |  '## Overview'
  'SPEC_pallet_relicyc_logypal1.pdf'  |  '## Cod. LOGYPAL1'
  'ART_relicyc_logypal1_datasheet_2021.pdf'  |  '## Internal LCA Results (2020, non-verified)'
  'EPD_pallet_relicyc_logypal1.pdf'  |  '## CONTENT DECLARATION'


### d) Unverified Supplier Claim (Tesa ECO Tape)

The tesa supplier brochure claims **68% CO₂ reduction** compared to conventional tape. This is a self-declared marketing claim, there is no independent EPD. The RAG should report the claim but flag that it is unverified.

In [17]:
QUERY = (
    "How much lower is the carbon footprint of tesa ECO tape compared to standard tape?"
)
print("---------------------------")
print(f"Query: {QUERY!r}")
print("---------------------------")
answer_claim = await ask(agent, QUERY)

2026-03-02 08:53:47.227 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:350 - Query: 'How much lower is the carbon footprint of tesa ECO tape compared to standard tape?'


---------------------------
Query: 'How much lower is the carbon footprint of tesa ECO tape compared to standard tape?'
---------------------------


2026-03-02 08:53:47.451 | INFO     | conversational_toolkit.embeddings.openai:get_embeddings:37 - OpenAI embeddings shape: (1, 1024)
2026-03-02 08:54:11.269 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:ask:353 - Answer:


Based on the provided information, it's challenging to provide a precise numerical comparison of carbon footprints between tesa ECO tape and standard tape due to the lack of independently verified Environmental Product Declarations (EPDs) for both products. However, here are some points from the sources:

1. tesa tesapack ECO tape claims a 68% reduction in CO₂ emissions compared to their 2019 baseline conventional product. This is based on an internal comparative assessment and hasn't been independently verified.
2. There's no EPD available for tesa ECO tape, making it difficult to compare its carbon footprint with standard tapes.

To determine the exact difference in carbon footprints between tesa ECO tape and standard tape, you would need to have access to independently verified EPDs for both products or contact PrimePack AG for further information.
Sources (5):
  'ART_supplier_brochure_tesa_ECO.pdf'  |  '## A New Standard in Sustainable Sealing'
  'REF_ecologo_catalogue.pdf'  |  '##

---

## Running the Full Pipeline in One Call

We have now gone through the pipeline step by step. For convenience, the `run_pipeline()` function executes all five steps end-to-end.

Use it for quick one-shot queries. Use the individual step functions above when you need
to inspect intermediate results or iterate on a specific stage.

In [18]:
from sme_kt_zh_collaboration_rag.feature0_baseline_rag import run_pipeline

answer = await run_pipeline(
    backend=BACKEND,
    query="What sustainability certifications do the pallets in the portfolio have?",
    reset_vs=False,
)
print(answer)

2026-03-02 08:54:37.693 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:run_pipeline:380 - Starting Baseline RAG pipeline
2026-03-02 08:54:37.695 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:run_pipeline:381 - backend='ollama'  model=None  max_files=5  reset_vs=False  top_k=5
2026-03-02 08:54:37.700 | WARNING  | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:199 - Skipping unsupported file type '': .DS_Store
2026-03-02 08:54:37.702 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:206 - Chunking 4 files from /Users/pkoerner/Desktop/Kanton_Zurich/sme-kt-zh-collaboration-rag/data


5


2026-03-02 08:54:44.958 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_internal_procurement_policy.pdf: 12 chunks
2026-03-02 08:54:47.469 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_logylight_incomplete_datasheet.pdf: 6 chunks
2026-03-02 08:54:50.189 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_product_catalog.pdf: 7 chunks
2026-03-02 08:54:50.219 | DEBUG    | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:225 -   ART_product_overview.xlsx: 1 chunks
2026-03-02 08:54:50.220 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:load_chunks:229 - Done, 26 chunks total
2026-03-02 08:54:50.220 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:inspect_chunks:239 - ------ Chunk inspection -------
2026-03-02 08:54:50.220 | INFO     | sme_kt_zh_collaboration_rag.feature0_baseline_rag:inspect_chunks:240 - Total chunks: 26; Source files:

OSError: sentence-transformers/text-embedding-3-small is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

---

## Summary

| Step | Function |
|---|---|
| 1. Load & chunk | `load_chunks(max_files)` |
| 2. Embed & index | `build_vector_store(chunks, emb, reset)` |
| 3. Inspect retrieval | `inspect_retrieval(query, vs, emb, top_k)` |
| 4. Build agent | `build_agent(vs, emb, llm, top_k, system_prompt, number_query_expansion)` |
| 5. Generate answer | `ask(agent, query, history)` |